In [0]:
import pandas as pd
from nba_api.stats.endpoints import playercareerstats
from nba_api.stats.static import players, teams

In [0]:
players_df = pd.DataFrame(players.get_players())
teams_df = pd.DataFrame(teams.get_teams())

In [0]:
print(f"Total players: {len(players_df)}")
print(f"Active players: {players_df['is_active'].sum()}")

In [0]:
import time
from nba_api.stats.endpoints import leaguedashplayerstats

# Define seasons from 2015-16 onwards
seasons = ['2015-16', '2016-17', '2017-18', '2018-19', '2019-20',
           '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']

print(f"Fetching player stats for {len(seasons)} seasons (2015-16 to 2024-25)...\n")

all_season_stats = []
errors = []

for season in seasons:
    try:
        print(f"Fetching season {season}...")
        
        # Fetch all player stats for this season
        stats = leaguedashplayerstats.LeagueDashPlayerStats(
            season=season,
            season_type_all_star='Regular Season'
        )
        
        season_data = stats.get_data_frames()[0]
        season_data['SEASON'] = season  # Add season identifier
        
        all_season_stats.append(season_data)
        print(f"  ✓ Found {len(season_data)} players for {season}")
        
        # Rate limiting - wait between API calls
        time.sleep(0.6)
        
    except Exception as e:
        errors.append({'season': season, 'error': str(e)})
        print(f"  ✗ Error fetching {season}: {str(e)[:100]}")
        continue

# Combine all season stats into one dataframe
if all_season_stats:
    season_stats = pd.concat(all_season_stats, ignore_index=True)
    print(f"\n✓ Successfully fetched stats for {len(seasons) - len(errors)} seasons")
    print(f"Total rows: {len(season_stats):,}")
    print(f"Unique players: {season_stats['PLAYER_ID'].nunique():,}")
    if errors:
        print(f"\nErrors encountered: {len(errors)}")
else:
    print("\n✗ No stats were fetched.")
    season_stats = pd.DataFrame()

In [0]:
# This data is already aggregated by player and season from the API
# Each row represents a player's stats for one season

if not season_stats.empty:
    # Select and rename key columns
    player_season_stats = season_stats[[
        'SEASON', 'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION',
        'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT',
        'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'PF', 'PTS',
        'PLUS_MINUS'
    ]].copy()
    
    # Sort by season and games played
    player_season_stats = player_season_stats.sort_values(['SEASON', 'GP'], ascending=[True, False])
    
    print(f"Player-Season Stats:")
    print(f"  Total records: {len(player_season_stats):,}")
    print(f"  Unique players: {player_season_stats['PLAYER_ID'].nunique():,}")
    print(f"  Seasons covered: {sorted(player_season_stats['SEASON'].unique())}")
    print(f"\nTop 10 players by games played in 2023-24:")
    display(player_season_stats[player_season_stats['SEASON'] == '2023-24'].head(10))
else:
    print("No season stats available to aggregate")
    player_season_stats = pd.DataFrame()

In [0]:
# Aggregate career totals across all seasons
if not season_stats.empty:
    career_totals = season_stats.groupby(['PLAYER_ID', 'PLAYER_NAME']).agg({
        'GP': 'sum',           # Total games played
        'W': 'sum',            # Total wins
        'L': 'sum',            # Total losses
        'MIN': 'sum',          # Total minutes
        'FGM': 'sum',          # Total field goals made
        'FGA': 'sum',          # Total field goals attempted
        'FG3M': 'sum',         # Total 3-pointers made
        'FG3A': 'sum',         # Total 3-pointers attempted
        'FTM': 'sum',          # Total free throws made
        'FTA': 'sum',          # Total free throws attempted
        'OREB': 'sum',         # Total offensive rebounds
        'DREB': 'sum',         # Total defensive rebounds
        'REB': 'sum',          # Total rebounds
        'AST': 'sum',          # Total assists
        'TOV': 'sum',          # Total turnovers
        'STL': 'sum',          # Total steals
        'BLK': 'sum',          # Total blocks
        'PF': 'sum',           # Total personal fouls
        'PTS': 'sum',          # Total points
        'PLUS_MINUS': 'sum',   # Total plus/minus
        'TEAM_ID': 'last'      # Most recent team
    }).reset_index()
    
    # Calculate career percentages and averages
    career_totals['W_PCT'] = (career_totals['W'] / career_totals['GP'] * 100).round(1)
    career_totals['FG_PCT'] = (career_totals['FGM'] / career_totals['FGA'] * 100).round(1)
    career_totals['FG3_PCT'] = (career_totals['FG3M'] / career_totals['FG3A'] * 100).round(1)
    career_totals['FT_PCT'] = (career_totals['FTM'] / career_totals['FTA'] * 100).round(1)
    career_totals['PPG'] = (career_totals['PTS'] / career_totals['GP']).round(1)
    career_totals['RPG'] = (career_totals['REB'] / career_totals['GP']).round(1)
    career_totals['APG'] = (career_totals['AST'] / career_totals['GP']).round(1)
    career_totals['MPG'] = (career_totals['MIN'] / career_totals['GP']).round(1)
    
    # Sort by total games played
    career_totals = career_totals.sort_values('GP', ascending=False)
    
    print(f"Career Totals (2015-16 onwards):")
    print(f"  Total players: {len(career_totals):,}")
    print(f"\nTop 10 players by games played:")
    display(career_totals[['PLAYER_NAME', 'GP', 'W_PCT', 'PPG', 'RPG', 'APG', 'MPG', 
                           'FG_PCT', 'FG3_PCT', 'FT_PCT', 'PTS']].head(10))
else:
    print("No season stats available to aggregate")
    career_totals = pd.DataFrame()